# 08 — Results Comparison and Visualisation

Load saved experiment JSON files and produce comparison plots.

**No API key required.**

In [ ]:
import sys; sys.path.insert(0, '..')
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.config import Settings

settings    = Settings()
results_dir = pathlib.Path(settings.results_dir)
result_files= sorted(results_dir.glob('*.json'))
print(f'Results dir : {results_dir}')
print(f'Result files: {len(result_files)}')

## 1. Load Results

In [ ]:
records=[]
for f in result_files:
    with open(f) as fp: records.append(json.load(fp))

# Fallback to illustrative data if no files found
if not records:
    print('No files found — using illustrative placeholder data.')
    records=[
        {'method':'Embedding only',   'mrr':0.312,'hits@1':0.221,'hits@3':0.353,'hits@10':0.497,'cost':0.000},
        {'method':'LLM (minimal)',    'mrr':0.341,'hits@1':0.248,'hits@3':0.381,'hits@10':0.521,'cost':0.012},
        {'method':'LLM (with_desc)',  'mrr':0.358,'hits@1':0.263,'hits@3':0.401,'hits@10':0.538,'cost':0.018},
        {'method':'LLM (strict)',     'mrr':0.349,'hits@1':0.255,'hits@3':0.392,'hits@10':0.528,'cost':0.022},
        {'method':'LLM (cot)',        'mrr':0.371,'hits@1':0.275,'hits@3':0.414,'hits@10':0.551,'cost':0.019},
        {'method':'RL (LinUCB)',      'mrr':0.383,'hits@1':0.287,'hits@3':0.428,'hits@10':0.562,'cost':0.021},
        {'method':'RL + Budget',      'mrr':0.391,'hits@1':0.295,'hits@3':0.437,'hits@10':0.571,'cost':0.015},
    ]
df=pd.DataFrame(records)
display(df)

## 2. Method Comparison: MRR & Hits@K

In [ ]:
metrics=['mrr','hits@1','hits@3','hits@10']
methods=df.get('method',df.index).tolist()
colors=plt.cm.tab10(np.linspace(0,1,len(df)))
fig,axes=plt.subplots(1,4,figsize=(16,5))
for ax,metric in zip(axes,metrics):
    vals=df[metric].values if metric in df.columns else np.zeros(len(df))
    bars=ax.bar(range(len(df)),vals,color=colors,edgecolor='white')
    ax.set(title=metric.upper(),ylabel='Score',xticks=[])
    ax.set_ylim(0,vals.max()*1.25 if vals.max()>0 else 1)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.004,
                f'{v:.3f}',ha='center',va='bottom',fontsize=8)
handles=[mpatches.Patch(color=colors[i],label=m) for i,m in enumerate(methods)]
fig.legend(handles=handles,loc='lower center',ncol=4,bbox_to_anchor=(0.5,-0.05))
plt.suptitle('Method Comparison on FB15k-237',y=1.02,fontsize=13)
plt.tight_layout(); plt.show()

## 3. Cost vs MRR Trade-off

In [ ]:
if 'cost' not in df.columns:
    df['cost']=0.0
fig,ax=plt.subplots(figsize=(8,5))
for i,row in df.iterrows():
    ax.scatter(row.get('cost',0),row.get('mrr',0),s=120,color=colors[i],zorder=3)
    ax.annotate(row.get('method',str(i)),
                (row.get('cost',0),row.get('mrr',0)),
                textcoords='offset points',xytext=(6,4),fontsize=8)
ax.set(title='Cost vs MRR Trade-off',xlabel='Cost (USD)',ylabel='MRR')
ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

## 4. RL Learning Curve

In [ ]:
rl_files=list(results_dir.glob('*bandit*.json'))+list(results_dir.glob('*rl*.json'))
if rl_files:
    with open(rl_files[0]) as f: rl_data=json.load(f)
    rewards=rl_data.get('reward_history',[])
    if rewards:
        fig,ax=plt.subplots(figsize=(9,4))
        ax.plot(np.cumsum(rewards),color='steelblue',label='Cumulative')
        w=max(1,len(rewards)//10)
        roll=np.convolve(rewards,np.ones(w)/w,mode='valid')
        ax.plot(range(w-1,len(rewards)),roll,color='coral',linewidth=2,label=f'Rolling avg w={w}')
        ax.set(title='RL Bandit Learning Curve',xlabel='Step',ylabel='Reward')
        ax.legend(); plt.tight_layout(); plt.show()
else:
    print('Run notebook 06 first to generate RL result files.')

## 5. Summary Table

In [ ]:
summary=df[[c for c in ['method']+metrics+['cost'] if c in df.columns]].copy()
if 'method' in summary.columns: summary=summary.set_index('method')
num_cols=[c for c in metrics if c in summary.columns]
print('=== Final Results Summary ===')
display(summary.style
        .highlight_max(axis=0,subset=num_cols,color='#d4f0c0')
        .format({c:'{:.4f}' for c in num_cols}))